# DnD class implementation

First, let's handle imports.

In [18]:
from enum import Enum
from pydantic import BaseModel, computed_field, Field
from typing import Annotated, List, Literal, Union

Let's get to work.

In [19]:
class Ability(str, Enum):
	STR = "Strength"
	DEX = "Dexterity"
	CON = "Constitution"
	INT = "Intelligence"
	WIS = "Wisdom"
	CHA = "Charisma"

In [20]:
class Size(str, Enum):
	SMALL = "Small"
	MEDIUM = "Medium"

In [21]:
class AbilityScores(BaseModel):
	strength: int = Field(ge=1, le=30, default=10)
	dexterity: int = Field(ge=1, le=30, default=10)
	constitution: int = Field(ge=1, le=30, default=10)
	intelligence: int = Field(ge=1, le=30, default=10)
	wisdom: int = Field(ge=1, le=30, default=10)
	charisma: int = Field(ge=1, le=30, default=10)

	def get_modifier(self, stat: int) -> int:
		return (stat - 10) // 2

Let's see how we'd create races.

In [22]:
class BaseRace(BaseModel):
	size: Size = Size.MEDIUM
	speed: int = 30
	languages: List[str] = ["Common"]

In [23]:
class Elf(BaseRace):
	race_type: Literal["Elf"] = "Elf"
	darvision_radius: int = 60
	fey_ancestry: bool = True
	trance: bool = True

In [24]:
class Dragonborn(BaseRace):
	race_type: Literal["Dragonborn"] = "Dragonborn"
	dragonic_ancestry: str
	breath_weapon_damage_type: str

In [25]:
DnDRace = Annotated[Union[Elf, Dragonborn], Field(discriminator="race_type")]

In [26]:
class BaseClassLevel(BaseModel):
	level: int = Field(ge=1, le=20, default=1)
	subclass: str | None = None

In [27]:
class Fighter(BaseClassLevel):
	class_type: Literal["Fighter"] = "Fighter"
	fighting_style: str

	def action_surges(self) -> int:
		if self.level >= 17:
			return 2
		if self.level >= 2:
			return 1
		return 0

In [28]:
class Wizard(BaseClassLevel):
	class_type: Literal["Wizard"] = "Wizard"
	spellbook: List[str] = Field(default_factory=list)
	prepared_spells: List[str] = Field(default_factory=list)

In [29]:
DnDClass = Annotated[Union[Fighter, Wizard], Field(discriminator="class_type")]

In [30]:
class Character(BaseModel):
	name: str
	base_stats: AbilityScores
	race: DnDRace
	classes: List[DnDClass] = Field(default_factory=list)

	@computed_field
	def total_level(self) -> int:
		return sum(c.level for c in self.classes)
	
	@computed_field
	def proficiency_bonus(self) -> int:
		return ((self.total_level - 1) // 4) + 2
	
	@computed_field
	def character_classes_summary(self) -> str:
		return " / ".join(f"{c.class_type} {c.level}" for c in self.classes)

Usage example.

In [31]:
character_data = {
	"name": "Galandriel",
	"base_stats": {
		"strength": 8,
		"dexterity": 16,
		"constitution": 12,
		"intelligence": 18,
		"wisdom": 10,
		"charisma": 14,
	},
	"race": {
		"race_type": "Elf",
		"darkvision_radius": 60,
		"languages": ["Common", "Elvish"],
	},
	"classes": [
		{
			"class_type": "Fighter",
			"level": 2,
			"fighting_style": "Defense",
		},
		{
			"class_type": "Wizard",
			"level": 3,
			"subclass": "School of Evocation",
			"spellbook": ["Mage Armor", "Fireball", "Shield"],
		},
	],
}

In [32]:
my_char = Character.model_validate(character_data)

print(f"Name: {my_char.name}")
print(f"Class: {my_char.character_classes_summary}")
print(f"Proficiency Bonus: +{my_char.proficiency_bonus}")
print(f"Action Surges Available: {my_char.classes[0].action_surges}")

Name: Galandriel
Class: Fighter 2 / Wizard 3
Proficiency Bonus: +3
Action Surges Available: <bound method Fighter.action_surges of Fighter(level=2, subclass=None, class_type='Fighter', fighting_style='Defense')>
